# CLOY Processing

This notebook is for you to work with the CLOY data folder.

In [26]:
import json
import numpy as np
import pandas as pd
from textblob import TextBlob

## 1. Load the data and look at one thread 


In [2]:
threads = []
with open("data/RedditExplorer/cloy_chronological.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        threads.append(json.loads(line))

In [3]:
print(len(threads))

3075


trying to go into the comments of the threads 

In [4]:
thread = threads[1]
print(thread.keys())
print("Title:", thread["title"])
print("comment numbers (top level)", len(thread["comments"]))


dict_keys(['post_id', 'timestamp', 'author', 'subreddit', 'title', 'selftext', 'url', 'score', 'permalink', 'comments'])
Title: ok, let’s get this started! What are your thoughts about the show?
comment numbers (top level) 10


In [5]:
firstComment = thread["comments"][0]
print(firstComment.keys())

dict_keys(['comment_id', 'parent_id', 'author', 'comment_body', 'score', 'created_utc', 'replies'])


i want to explore the comment body and sift through replies 

In [6]:
print(firstComment["comment_body"])
print()


i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.



In [7]:
print(firstComment["replies"][0]["comment_body"])
print(firstComment["replies"][0]["replies"][0]["comment_body"])

Did you have a favorite episode? Mine would have to be ep. 9 ☺️
I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.


## Getting text 
so now i want to get the text from the thread now that i figured out how to get through the replies. I think since I dont know the amount of replies in a thread, I want to recursively go through this comment replies and stop once i get to the bottom.

In [8]:
def commentsText(comments):
    text = ''
    for comment in comments:
        text = text + ' ' + comment["comment_body"]
        if "replies" in comment:
            text = text + ' ' +commentsText(comment["replies"]) 
    return text

In [9]:
print(commentsText(thread["comments"]))

 i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.  Did you have a favorite episode? Mine would have to be ep. 9 ☺️  I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless. I love this show. I stumbled across it on Netflix, gave it a try, and fell in love. I finished it this morning and already miss it. I love Yoon Se-Ri and Jeong Hyeok and nearly all of the other characters. I was so happy with the ending. I’ve added two more kdramas to my Netflix list.  Which ones!  Itaewon Class, Chocolate, When the Camellia Blooms  Thank you :) I'm currently obsessed with Hyun Bin. I've never watched Korean dramas, the most as

i want to also get the threads from the comments themselves so id want to call the previous function in this to make it nested and text blob will want to count this


i want to add the commentbody to text and to call tiself on the replies (recursion again omg i cant believe it lol)

In [10]:
def threadText(thread):
    text = thread["title"] + thread["selftext"] + commentsText(thread["comments"])
    text=text.replace('’', "'")
    return text.lower()


print(threadText(threads[1])[:500])

ok, let's get this started! what are your thoughts about the show?i haven't seen a kdrama for the longest time and it was actually my dad who chose to stream cloy on netflix. he stopped watching after the second episode but i went on after immediately falling in love with the leads. hyun bin is just so charismatic as capt. ri jeong hyeok and son ye-jin is perfect as yoon se-ri. she's definitely got the rom-com and drama genre down pat! i also loved capt. ri's squad, most especially pyo chi su ☺️


## Count words in threads using textblob 
want to use the text blob dictionary!!! 

In [11]:
count = TextBlob(threadText(threads[1])).word_counts

sorted1 = sorted(count.items(), key=lambda pair: pair[1], reverse=True)
print(sorted1[:10])

[('i', 91), ('the', 62), ('and', 49), ('to', 36), ('a', 35), ('was', 29), ('it', 28), ('my', 26), ('in', 25), ('so', 21)]


now i want to count the numbher of threads 

In [12]:
threadIDs = []
titles = []
wordCount = []

for thread in threads:
    threadIDs.append(thread["post_id"])
    titles.append(thread["title"])
    wordCount.append(TextBlob(threadText(thread)).word_counts)

print(len(wordCount))

3075


## make vectors for cosine sim
each word count dictonary has a different get of words in differing orders so you want to put every thread on the same shared list 

so we want to get the vocab in a list and then make vectors to fill in the numbers. 


In [13]:
def vocab(wordCount):
    allWords = set()
    for counts in wordCount:
        allWords.update(counts.keys())
    return sorted(allWords)


voc = vocab(wordCount)
print(len(voc), "words in vocab")
print(voc[5000:5020])

37265 words in vocab
['callign', 'calligraphy', 'callin', 'calling', 'callous', 'callousness', 'callout', 'calls', 'calm', 'calmed', 'calmer', 'calmest', 'calming', 'calmly', 'calmness', 'calms', 'calmwife', 'calories', 'calves', 'calvino']


In [15]:
print("crash" in voc)

True


now after getting the vector i want to make a frequency table style vector. 

In [16]:
def vectors(wordCount, voc):
    position = {}
    for i, word in enumerate(voc):
        position[word] = i
    vector1 = np.zeros((len(wordCount), len(voc)))
    for row, counts in enumerate(wordCount):
        for word, count in counts.items():
            vector1[row, position[word]] = count
    return vector1
    
vec = vectors(wordCount, voc)
print(vec.shape)

(3075, 37265)


try to make one row and one column per term 

In [17]:
listofwords =["captain", "episode", "hyun", "crash", "love", "seri", "watch"]
columns = [voc.index(word) for word in listofwords]
pd.DataFrame(vec[:5, columns], index=threadIDs[:5], columns=listofwords)



,captain,episode,hyun,crash,love,seri,watch
dkxewd,0.0,0.0,0.0,1.0,0.0,0.0,0.0
ffh52z,0.0,6.0,3.0,0.0,10.0,2.0,11.0
fguorr,0.0,0.0,0.0,0.0,0.0,1.0,0.0
fhcrpl,2.0,0.0,0.0,0.0,3.0,1.0,0.0
fj5byj,0.0,2.0,1.0,1.0,9.0,4.0,2.0


## Cosine calculation 


In [29]:
def cosineSim(vec1, vec2):
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    norm1 = np.sqrt(np.sum(v1*v1))
    norm2 = np.sqrt(np.sum(v2*v2))
    cosine = np.dot(v1, v2) / (norm1 * norm2)
    return cosine

In [30]:
print(titles[1], "\n vs", titles[2])
print(cosineSim(vec[1], vec[2]))

ok, let’s get this started! What are your thoughts about the show? 
 vs Find someone who’ll make you noodles just like Ri Jeong Hyeok-ssi
0.520705576592439
